In [1]:
def converg_check(opts, lc, angleA, sd_iter=None):
    """
    Checks convergence criteria and generates a message for display.

    Parameters:
    - opts: Dictionary containing options for convergence checks (e.g., minangle, earlystop, rmsstop, cfstop).
    - lc: Dictionary with lists for convergence parameters, including 'prms', 'rms', and 'cost'.
    - angleA: Float representing the angle between subspaces.
    - sd_iter: Optional integer for checking if a slowing-down stop is required (defaults to None).

    Returns:
    - convmsg: String message indicating the convergence status.
    """

    convmsg = ""

    # Check angle convergence criterion
    if angleA < opts.get('minangle', float('inf')):
        convmsg = f"Convergence achieved (angle between subspaces smaller than {opts['minangle']:.2e})\n"

    # Check early stopping criterion
    elif opts.get('earlystop', False) and lc['prms'][-1] > lc['prms'][-2]:
        convmsg = "Early stopping.\n"

    # Check RMS stop criterion
    elif 'rmsstop' in opts and opts['rmsstop'] and len(lc['rms']) - 1 > opts['rmsstop'][0]:
        numiter = opts['rmsstop'][0]
        abs_tol = opts['rmsstop'][1]
        rel_tol = opts['rmsstop'][2] if len(opts['rmsstop']) > 2 else None

        rms1 = lc['rms'][-numiter - 1]
        rms2 = lc['rms'][-1]

        if abs(rms1 - rms2) < abs_tol or (rel_tol is not None and abs((rms1 - rms2) / rms2) < rel_tol):
            convmsg = f"Stop: RMS does not change much for {numiter} iterations.\n"

    # Check cost function stop criterion
    elif 'cfstop' in opts and opts['cfstop'] and len(lc['cost']) - 1 > opts['cfstop'][0]:
        numiter = opts['cfstop'][0]
        abs_tol = opts['cfstop'][1]
        rel_tol = opts['cfstop'][2] if len(opts['cfstop']) > 2 else None

        cost1 = lc['cost'][-numiter - 1]
        cost2 = lc['cost'][-1]

        if abs(cost1 - cost2) < abs_tol or (rel_tol is not None and abs((cost1 - cost2) / cost2) < rel_tol):
            convmsg = f"Stop: Cost does not change much for {numiter} iterations.\n"

    # Slowing-down stop criterion if sd_iter is provided
    elif sd_iter is not None and sd_iter == 40:
        convmsg = "Slowing-down stop. You may continue by changing the gradient type.\n"

    return convmsg


In [7]:
def test_angle_criterion_met():
    # Test 1 - Angle Criterion Met
    opts = {'minangle': 0.02, 'earlystop': False, 'rmsstop': None, 'cfstop': None}
    lc = {'prms': [1, 2], 'rms': [], 'cost': []}
    angleA = 0.01
    sd_iter = None

    # Expected output
    expected = 'Convergence achieved (angle between subspaces smaller than 2.00e-02)'

    # Run the Python version of converg_check
    convmsg = converg_check(opts, lc, angleA, sd_iter)
    
    # Strip whitespace for comparison
    expected_trimmed = expected.strip()
    convmsg_trimmed = convmsg.strip()

    # Display results
    print('Test 1 - Angle Criterion Met')
    print(f'Expected: {expected}')
    print(f'Got: {convmsg}')

    # Compare using stripped strings
    if expected_trimmed == convmsg_trimmed:
        print('Result: Passed\n')
    else:
        print('Result: Failed\n')

# Run Test 1
test_angle_criterion_met()


Test 1 - Angle Criterion Met
Expected: Convergence achieved (angle between subspaces smaller than 2.00e-02)
Got: Convergence achieved (angle between subspaces smaller than 2.00e-02)

Result: Passed



In [8]:
def test_angle_criterion_not_met():
    # Test 2 - Angle Criterion Not Met
    opts = {'minangle': 0.02, 'earlystop': False, 'rmsstop': None, 'cfstop': None}
    lc = {'prms': [1, 2], 'rms': [], 'cost': []}
    angleA = 0.03  # Angle is above the minangle, so criterion is not met
    sd_iter = None

    # Expected output (should be empty)
    expected = ''

    # Run the Python version of converg_check
    convmsg = converg_check(opts, lc, angleA, sd_iter)
    
    # Display results
    print('Test 2 - Angle Criterion Not Met')
    print(f'Expected: "{expected}"')
    print(f'Got: "{convmsg}"')

    # Compare output
    if convmsg == expected:
        print('Result: Passed\n')
    else:
        print('Result: Failed\n')

# Run Test 2
test_angle_criterion_not_met()


Test 2 - Angle Criterion Not Met
Expected: ""
Got: ""
Result: Passed



In [9]:
def test_early_stopping_met():
    # Test 3 - Early Stopping Met
    # This test verifies if the early stopping condition in converg_check
    # is triggered correctly based on the values in opts and lc.

    # Define the options and lc structures
    opts = {'minangle': 0, 'earlystop': True, 'rmsstop': None, 'cfstop': None}
    lc = {'prms': [1.5, 1.4, 1.6], 'rms': [], 'cost': []}
    angleA = 0.03
    sd_iter = None

    # Expected output
    expected = 'Early stopping.'

    # Run the Python version of converg_check
    convmsg = converg_check(opts, lc, angleA, sd_iter)

    # Display the test description and results
    print('Test 3 - Early Stopping Met')
    print(f'Expected: {expected}')
    print(f'Got: {convmsg}')

    # Trim whitespace for comparison
    expected_trimmed = expected.strip()
    convmsg_trimmed = convmsg.strip()

    # Check if the result matches the expected output
    if convmsg_trimmed == expected_trimmed:
        print('Result: Passed\n')
    else:
        print('Result: Failed\n')

# Run the test
test_early_stopping_met()


Test 3 - Early Stopping Met
Expected: Early stopping.
Got: Early stopping.

Result: Passed



In [10]:
def test_rms_criterion_absolute_tolerance_met():
    # Test 4 - RMS Criterion - Absolute Tolerance Met
    # This test verifies if the RMS criterion for absolute tolerance is
    # correctly triggered based on the values in opts and lc.

    # Define the options and lc dictionaries
    opts = {'minangle': 0, 'earlystop': False, 'rmsstop': [3, 0.01], 'cfstop': None}
    lc = {'prms': [], 'rms': [0.15, 0.14, 0.14, 0.141, 0.141], 'cost': []}
    angleA = 0.03
    sd_iter = None

    # Expected output
    expected = 'Stop: RMS does not change much for 3 iterations.'

    # Run the converg_check function
    convmsg = converg_check(opts, lc, angleA, sd_iter)

    # Display the test description and results
    print('Test 4 - RMS Criterion - Absolute Tolerance Met')
    print(f'Expected: {expected}')
    print(f'Got: {convmsg}')

    # Trim whitespace for comparison
    expected_trimmed = expected.strip()
    convmsg_trimmed = convmsg.strip()

    # Check if the result matches the expected output
    if convmsg_trimmed == expected_trimmed:
        print('Result: Passed\n')
    else:
        print('Result: Failed\n')

# Run the test
test_rms_criterion_absolute_tolerance_met()


Test 4 - RMS Criterion - Absolute Tolerance Met
Expected: Stop: RMS does not change much for 3 iterations.
Got: Stop: RMS does not change much for 3 iterations.

Result: Passed



In [11]:
def test_cost_criterion_absolute_tolerance_met():
    # Test 5 - Cost Criterion - Absolute Tolerance Met
    # This test verifies if the Cost criterion for absolute tolerance is
    # correctly triggered based on the values in opts and lc.

    # Define the options and lc dictionaries
    opts = {'minangle': 0, 'earlystop': False, 'rmsstop': None, 'cfstop': [2, 0.01]}
    lc = {'prms': [], 'rms': [], 'cost': [0.5, 0.49, 0.495, 0.495]}
    angleA = 0.03
    sd_iter = None

    # Expected output
    expected = 'Stop: Cost does not change much for 2 iterations.'

    # Run the converg_check function
    convmsg = converg_check(opts, lc, angleA, sd_iter)

    # Display the test description and results
    print('Test 5 - Cost Criterion - Absolute Tolerance Met')
    print(f'Expected: {expected}')
    print(f'Got: {convmsg}')

    # Trim whitespace for comparison
    expected_trimmed = expected.strip()
    convmsg_trimmed = convmsg.strip()

    # Check if the result matches the expected output
    if convmsg_trimmed == expected_trimmed:
        print('Result: Passed\n')
    else:
        print('Result: Failed\n')

# Run the test
test_cost_criterion_absolute_tolerance_met()


Test 5 - Cost Criterion - Absolute Tolerance Met
Expected: Stop: Cost does not change much for 2 iterations.
Got: Stop: Cost does not change much for 2 iterations.

Result: Passed

